# Notebook 2 / 2 — Cải tiến: **TabPFN-GBX-PHE**

**Kaggle:** Accelerator = **GPU T4 x2**, Internet = **On**. Thời gian ước tính: ~3–4 giờ
(phương pháp của ta trên GPU 0, baseline fine-tune trên GPU 1, chạy song song).

### Điểm yếu của TabPFN v2 mà ta nhắm vào
Phần A cho thấy TabPFN v2 thắng tổng thể nhưng **thua mô hình cây ở các dataset có nhiều biến
phân loại** (cylinder-bands, credit-g, credit-approval, dresses-sales, ...):
1. Biến phân loại đi vào TabPFN dưới dạng mã số nguyên tùy ý; prior giả lập chủ yếu là cơ chế liên
   tục nên mô hình khó học thống kê theo từng giá trị từ vài trăm dòng.
2. Prior cho TabPFN thiên kiến trơn; GBDT có thiên kiến bổ sung (split theo trục, target statistics).
3. Một cấu hình tiền xử lý cố định không tối ưu cho mọi dataset.

### Phương pháp (TabPFN v2 giữ nguyên, không fine-tune)
1. **TabPFN-GBX**: làm giàu ngữ cảnh bằng cross-fitting K-fold: **OOF target encoding** cho biến
   phân loại và **OOF log-xác suất CatBoost**. Dòng train nhận giá trị out-of-fold (không rò rỉ nhãn),
   dòng test nhận trung bình của K mô hình fold.
2. **Portfolio cấu hình TabPFN v2** (cùng checkpoint): one-hot / ordinal cho biến phân loại, biến đổi
   KDI, đặc trưng đa thức, nhiệt độ softmax, trung bình logits, 32 estimator; thêm TabPFN-GBX với
   one-hot.
3. **Post-hoc ensemble selection** (Caruana 2004) trên toàn bộ portfolio + CatBoost, trọng số chọn
   trên validation nội bộ (80/20 trong fold train, không dùng tập test) = **TabPFN-GBX-PHE**.

### So sánh
* Bản gốc: **TabPFN v2**.
* Đã có: **TabPFN v2 + fine-tune** (module chính thức), **Ens(TabPFN v2, CatBoost)**,
  **Ens(TabPFN v2-FT, CatBoost)**, **TabPFN v2-PHE** (chỉ ensemble cấu hình TabPFN, ý tưởng PHE của
  paper Nature), CatBoost.
* Của ta: **TabPFN-GBX**, **TabPFN-GBX-Ens**, **TabPFN-GBX-PHE** (bản chính), **+FT**; ablation:
  chỉ TE, chỉ logits GBDT. So với TabPFN v2-PHE cho thấy phần GBX đóng góp thêm bao nhiêu.

## 0. Cài đặt chung

In [ ]:
# >>>>> SỬA DÒNG NÀY: URL repo GitHub của bạn <<<<<
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/tabpfn-dl-project.git"

import os
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO = os.path.join(WORK, "tabpfn-dl-project")
os.environ["OPENML_CACHE"] = os.path.join(WORK, "openml_cache")
print(WORK, REPO)

In [ ]:
!nvidia-smi -L
if not os.path.isdir(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !cd {REPO} && git pull -q
%cd {REPO}
!pip install -q -r requirements.txt
!python tests/test_smoke.py

In [ ]:
import torch, tabpfn, subprocess, time, sys, glob, shutil, zipfile
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| GPUs", torch.cuda.device_count(), "| tabpfn", tabpfn.__version__)
os.makedirs("logs", exist_ok=True)

def run(cmd, log):
    """Chạy lệnh shell, in log ra cell và ghi vào file."""
    print(">>", cmd)
    with open(log, "a") as f:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            sys.stdout.write(line); f.write(line)
        p.wait()
    if p.returncode != 0:
        print(f"!! exit code {p.returncode} (xem {log})")

def run_parallel(jobs):
    """jobs = [(cmd, gpu_id, logfile), ...] -> mỗi GPU một tiến trình, chờ tất cả."""
    procs = []
    for cmd, gpu, log in jobs:
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        print(f">> [GPU {gpu}] {cmd}")
        procs.append((subprocess.Popen(cmd, shell=True, env=env, stdout=open(log, "a"),
                                       stderr=subprocess.STDOUT), log))
    while any(p.poll() is None for p, _ in procs):
        time.sleep(120)
        for _, log in procs:
            lines = [l for l in open(log).read().splitlines() if l.strip()]
            print(f"[{log}] {lines[-1] if lines else ''}")
    print("all jobs finished")

## 1. Khôi phục kết quả Notebook 1

Upload `session1_results.zip` thành Kaggle Dataset (*Add Input → Upload*), rồi chạy cell dưới.
Nếu không gắn, cell sẽ chạy lại TabPFN v2 + các baseline (~1 giờ).

In [ ]:
BENCH = "results/benchmark"
zips = glob.glob("/kaggle/input/**/session1_results.zip", recursive=True)
found = glob.glob("/kaggle/input/**/results/benchmark/metrics.jsonl", recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(REPO); print("restored from", zips[0])
elif found:
    shutil.copytree(os.path.dirname(found[0]), BENCH, dirs_exist_ok=True)
    print("restored from", found[0])
else:
    print("Không có kết quả Notebook 1 -> chạy lại Phần A")
    run(f"python scripts/run_benchmark.py --source openml --max_rows 3000 --n_folds 3 "
        f"--out {BENCH} --val_preds --methods logreg,knn,rf,xgb,lgbm,catboost,tabpfn_v2",
        "logs/partA.log")
print(len(glob.glob(f"{BENCH}/preds/*.npz")), "prediction files")

## 2. Chạy song song: phương pháp của ta (GPU 0) và fine-tune baseline (GPU 1)

* GPU 0: các biến thể GBX (`tabpfn_te`, `tabpfn_gbx`, `tabpfn_gbx_te`, `gbx_onehot`; CatBoost OOF
  chạy trên CPU và dùng chung) và 7 cấu hình portfolio `pfn_*`.
* GPU 1: `tabpfn_v2_ft` (fine-tune chính thức, `FT_TIME` giây mỗi lần fit), ghi vào thư mục riêng rồi gộp lại.

Cell in dòng log cuối của mỗi tiến trình sau mỗi 2 phút.

In [ ]:
GBX_FOLDS, GBX_ITERS = 5, 500     # K của cross-fitting, số cây CatBoost
FT_TIME = 45                      # giây fine-tune mỗi lần fit (30 = nhanh hơn)
BENCH_FT = "results/benchmark_ft"
common = "--source openml --max_rows 3000 --n_folds 3 --val_preds"
cmd_ours = (f"python scripts/run_benchmark.py {common} --out {BENCH} "
            f"--methods tabpfn_te,tabpfn_gbx,tabpfn_gbx_te,gbx_onehot,"
            f"pfn_onehot,pfn_ordinal,pfn_kdi,pfn_poly,pfn_sharp,pfn_avgsm,pfn_wide "
            f"--gbx_folds {GBX_FOLDS} --gbx_iterations {GBX_ITERS}")
cmd_ft = (f"python scripts/run_benchmark.py {common} --out {BENCH_FT} "
          f"--methods tabpfn_v2_ft --ft_time_limit {FT_TIME}")
if torch.cuda.device_count() >= 2:
    run_parallel([(cmd_ours, 0, "logs/partB_ours.log"), (cmd_ft, 1, "logs/partB_ft.log")])
else:
    run(cmd_ours, "logs/partB_ours.log")
    run(cmd_ft, "logs/partB_ft.log")
!grep -c "ERR" logs/partB_ours.log logs/partB_ft.log || true

In [ ]:
# gộp kết quả fine-tune vào thư mục benchmark chính
import json
os.makedirs(f"{BENCH}/preds", exist_ok=True)
for f in glob.glob(f"{BENCH_FT}/preds/*.npz"):
    shutil.copy(f, f"{BENCH}/preds/")
done = set()
if os.path.exists(f"{BENCH}/metrics.jsonl"):
    for l in open(f"{BENCH}/metrics.jsonl"):
        r = json.loads(l); done.add((r["dataset"], r["fold"], r["method"], r.get("status")))
n = 0
if os.path.exists(f"{BENCH_FT}/metrics.jsonl"):
    with open(f"{BENCH}/metrics.jsonl", "a") as out:
        for l in open(f"{BENCH_FT}/metrics.jsonl"):
            r = json.loads(l)
            if (r["dataset"], r["fold"], r["method"], r.get("status")) not in done:
                out.write(l if l.endswith("\n") else l + "\n"); n += 1
print("merged", n, "fine-tune rows")

## 3. Ensemble selection (vài giây, CPU)

* `ens_pfn_cat` = TabPFN v2 + CatBoost (đã có)
* `ens_ft_cat` = TabPFN v2-FT + CatBoost (đã có)
* `ens_phe` = TabPFN v2 + 7 cấu hình `pfn_*` (PHE, ý tưởng đã có)
* `ens_gbx` = TabPFN v2 + TabPFN-GBX + CatBoost (**của ta**)
* `ens_gbx_phe` = portfolio + TabPFN-GBX (+ one-hot) + CatBoost (**của ta, bản chính**)
* `ens_gbx_phe_ft` = như trên + TabPFN v2-FT (**của ta**)

In [ ]:
run(f"python scripts/run_ensembles.py --bench {BENCH}", "logs/partB_ens.log")

## 4. Báo cáo cuối + đóng gói

Gồm: bảng Phần A, bảng Phần B (ΔAUC, W/T/L, Wilcoxon so với TabPFN v2 gốc), thay đổi accuracy và
log-loss, **phân tích theo nhóm** (dataset TabPFN v2 đang thua / đang thắng, có / không có biến phân
loại), biểu đồ **mức cải thiện theo độ yếu của TabPFN v2**, kết quả từng thành viên portfolio,
trọng số ensemble.

In [ ]:
!python scripts/make_report.py --bench {BENCH} --out results/report
!cd {REPO} && zip -qr {WORK}/final_results.zip results logs
from IPython.display import Markdown, display, Image
display(Markdown(open("results/report/report.md").read()))
for f in sorted(glob.glob("results/report/*.png")):
    display(Image(f))
print("-> TẢI VỀ:", f"{WORK}/final_results.zip")